# Chapter 10 — Network models

Companion notebook to `docs/models/ch10_network.md` and the Streamlit page `app/pages/10_Network_Models.py`.

Four canonical network families:
- **Erdős–Rényi** — pure randomness (null model).
- **Watts–Strogatz** — small-world (clustered ring lattice + a few rewirings).
- **Barabási–Albert** — scale-free (preferential attachment).
- **Random geometric** — spatial (link nearby nodes).

And three signature results:
- The **friendship paradox** — your friends have more friends than you do.
- **Distinct degree distributions** — Poisson vs power-law vs narrow.
- **Robustness** — hub-heavy networks tolerate random failures but collapse under targeted attack.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from modelthinker.networks.network import (
    build_barabasi_albert,
    build_erdos_renyi,
    build_geographic,
    build_watts_strogatz,
    degree_sequence,
    friendship_paradox,
    network_stats,
    robustness_curve,
)

## 1. The four families side by side

Build one graph from each family with roughly matched mean degree and compare their summary statistics. Watch how the shape of the degree distribution alone tells you what family a graph came from.

In [ ]:
n = 300
graphs = {
    'Erdős-Rényi':      build_erdos_renyi(n=n, p=4/(n-1),  seed=1),
    'Watts-Strogatz':   build_watts_strogatz(n=n, k=4, p=0.1, seed=1),
    'Barabási-Albert':  build_barabasi_albert(n=n, m=2,       seed=1),
    'Random geometric': build_geographic(n=n, radius=0.075,   seed=1),
}

rows = []
for name, G in graphs.items():
    s = network_stats(G)
    rows.append((name, s.mean_degree, s.mean_clustering,
                 s.mean_shortest_path, s.diameter, s.largest_component_frac))

print(f'{"Family":<20}  {"<k>":>6}  {"clust":>6}  {"<L>":>6}  {"D":>4}  {"LCC%":>6}')
print('-' * 60)
for name, k, c, L, D, lcc in rows:
    print(f'{name:<20}  {k:>6.2f}  {c:>6.3f}  {L:>6.2f}  {D:>4}  {100*lcc:>5.0f}%')

**Degree distributions — the fingerprint of each family.**

Poisson-ish for ER, sharply peaked around $k$ for WS, heavy-tailed for BA, roughly Poisson for RGG.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
colours = ['#4C6EF5', '#37B24D', '#F76707', '#7048E8']
for ax, (name, G), c in zip(axes, graphs.items(), colours, strict=True):
    d = degree_sequence(G)
    ax.hist(d, bins=max(3, d.max() + 1), color=c, alpha=0.85)
    ax.set(title=name, xlabel='degree')
axes[0].set_ylabel('count')
plt.tight_layout(); plt.show()

# Log-log for the BA family — the classic power-law fingerprint
ba_deg = degree_sequence(graphs['Barabási-Albert'])
vals, counts = np.unique(ba_deg, return_counts=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(vals, counts, 'o', color='#F76707')
ax.set(xlabel='degree k', ylabel='number of nodes',
       title='Barabási-Albert degree distribution (log-log)')
ax.grid(True, which='both', alpha=0.3); plt.show()

## 2. The friendship paradox

Feld (1991) proved that in **any** non-regular network, $\langle k \rangle_{\text{neighbours}} \ge \langle k \rangle$. Popular people appear in more friend lists, so they get sampled disproportionately. This isn't sociology — it's combinatorics.

In [ ]:
print(f'{"Family":<20}  {"<k>":>6}  {"<k_nbrs>":>10}  {"ratio":>7}  {"% beaten":>10}')
print('-' * 62)
for name, G in graphs.items():
    fp = friendship_paradox(G)
    ratio = fp.mean_neighbour_degree / max(fp.mean_degree, 1e-9)
    print(f'{name:<20}  {fp.mean_degree:>6.2f}  {fp.mean_neighbour_degree:>10.2f}  '
          f'{ratio:>7.2f}  {100*fp.fraction_beaten:>9.1f}%')

The scatter below plots each node's own degree against the mean degree of its neighbours, for the scale-free network. The dotted line is parity — every point above it is a person whose friends have more friends than they do. The cloud sits well above the line.

In [ ]:
G = graphs['Barabási-Albert']
fp = friendship_paradox(G)
valid = ~np.isnan(fp.per_node_neighbour_mean)

fig, ax = plt.subplots(figsize=(6, 5))
hi = max(fp.per_node_degree.max(), np.nanmax(fp.per_node_neighbour_mean))
ax.plot([0, hi], [0, hi], color='#868E96', ls=':', label='parity (y = x)')
ax.scatter(fp.per_node_degree[valid], fp.per_node_neighbour_mean[valid],
           color='#4C6EF5', alpha=0.6, s=30)
ax.set(xlabel='your degree', ylabel='mean degree of your friends',
       title=f'Friendship paradox on BA network — {100*fp.fraction_beaten:.0f}% beaten')
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 3. Robustness — random failures vs targeted attacks

Remove nodes one batch at a time and track the largest connected component. Random removal is a stand-in for random failures. Targeted removal always takes out the current highest-degree node. Scale-free networks have a dramatic **gap** between the two curves.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, (name, G) in zip(axes, graphs.items(), strict=True):
    rand = robustness_curve(G, strategy='random',   n_steps=30, seed=7)
    targ = robustness_curve(G, strategy='targeted', n_steps=30)
    ax.plot(100*rand.fractions_removed, 100*rand.largest_component_frac,
            color='#4C6EF5', lw=2.5, label='random failure')
    ax.plot(100*targ.fractions_removed, 100*targ.largest_component_frac,
            color='#E03131', lw=2.5, label='targeted (hubs)')
    ax.axhline(50, color='#868E96', ls=':', lw=1)
    ax.set(title=name, xlabel='% nodes removed', xlim=(0, 100), ylim=(0, 105))
axes[0].set_ylabel('largest CC (% of original)')
axes[0].legend(loc='lower left', fontsize=9)
plt.tight_layout(); plt.show()

print('Critical removal fractions (LCC drops below 50 %):')
for name, G in graphs.items():
    rand = robustness_curve(G, strategy='random',   n_steps=30, seed=7)
    targ = robustness_curve(G, strategy='targeted', n_steps=30)
    print(f'  {name:<20}  random {100*rand.critical_removal_frac:>4.0f}%   '
          f'targeted {100*targ.critical_removal_frac:>4.0f}%   '
          f'gap {100*(rand.critical_removal_frac - targ.critical_removal_frac):>+4.0f}%')

## 4. What-ifs to try

1. **The small-world transition.** Sweep the rewiring probability $p$ in WS from 0 to 1 in log-space and plot the average shortest path length. Where does the transition happen? (Answer: even a *tiny* $p$ collapses paths while clustering stays high.)
2. **When does BA lose its heavy tail?** Increase $m$ from 1 to 8. The tail flattens as $m$ grows because early nodes lose their preferential-attachment advantage.
3. **The percolation threshold for ER.** Set $p = c / (n - 1)$ and sweep $c$ from 0.5 to 3.0. Watch the largest connected component take off exactly at $c = 1$. That's the classic ER phase transition — Erdős' original result.
4. **Friendship paradox on a torus.** Compare `build_geographic` (random points in the unit square) against a regular grid graph (`networkx.grid_2d_graph`). The grid is 4-regular → no paradox. The random-geometric version → paradox lite.